# Appendix 2 — Numerical diffraction of finite grating illumination

**Advanced numerical experiment — prescribed finite-grating boundary fields**

A finite laser beam illuminates only part of a grating. The transmitted field then has an envelope, and different angular components can separate as they propagate. We will follow that field to a detector and inspect where two selected contributions overlap.

## Motivation and prerequisites

This appendix calculates an outgoing field by propagating a prescribed illuminated binary-mask boundary with FFT/angular-spectrum methods. In [7.3](7_3_Finite_Diffraction_Beams_and_Experimental_Recombination.ipynb), we assigned analytical Gaussian beams to order directions. Here we prescribe the illuminated binary-mask boundary and calculate the outgoing field from it. These are different physical models: neither is a rigorous material-grating scattering solver. Review the analytical lesson first for beam separation and overlap; use this appendix to study spectral broadening, filtering and numerical convergence.

In [7.2](7_2_Interference_of_Diffraction_Orders_from_a_Grating.ipynb), two indefinitely extended order fields produced fringes at every detector position. Their direction arrows separated, but their fields did not occupy separate narrow corridors. A laboratory laser illuminates a finite part of a grating. The outgoing field then has an envelope, and propagation can carry different parts of its angular spectrum into different spatial regions.

This lesson follows those fields from the grating to a movable detector. First inspect the entire transmitted field, then retain two spectral regions and compare their individual intensities with their coherent sum. The aim is to connect finite illumination, diffraction and spatial overlap without treating separation as loss of coherence.

Review [1.2: ideal waves and finite beams](1_2_Ideal_Plane_Waves_versus_Finite_Optical_Beams.ipynb), [2.3: allowed orders](2_3_Allowed_Diffraction_Orders.ipynb), [6.2: detector intensity](6_2_One_Order_Across_a_Detector.ipynb) and [7.1: mutual coherence](7_1_Interference_and_Mutual_Coherence_of_Two_Plane_Waves.ipynb). [0.4](0_4_Propagation_of_Scalar_Optical_Fields.ipynb) explains the propagation method. Run cells in order; no previous notebook needs to execute.

## Configuration and idealizations

The grating lies at $z=0$, repeats along $x$ with pitch $d$, and is uniform along $y$. Light arrives from $z>0$ and transmission travels toward negative $z$. Use wavelength in the homogeneous medium, lengths in micrometres, common polarization and time dependence $e^{-i\omega t}$. The detector lies at $z_D=-s$, where depth $s\geq0$.

The starting values are $\lambda=1$ µm, $d=4$ µm, envelope radius $w_0=8$ µm and normal incidence. Half of each period is transparent and half opaque. This ideal thin amplitude mask prescribes a transmitted boundary field; it does not solve reflection, absorption or electromagnetic scattering by a material grating. All intensities are relative scalar values with incident peak amplitude one.

## 1. Replace infinite illumination with a finite envelope

Compare the two prescribed incident fields at the same grating:

$$U_i^{\mathrm{plane}}(x)=A_0e^{ik_{x,i}x},\qquad
U_i(x)=A_0e^{-x^2/w_0^2}e^{ik_{x,i}x}.$$

The radius $w_0$ is where field amplitude falls to $1/e$, and intensity to $1/e^2$, of its on-axis value. It is measured across $x$ at the boundary. The Gaussian has tails rather than sharp edges. Here it is a prescribed boundary profile, not automatically an exact freely propagating Gaussian solution.

**Predict:** if illumination narrows while pitch stays fixed, do the ideal order centers move, or does each spectral peak broaden? In Panel A, compare the uniform reference line with the finite Gaussian and the same binary mask. The adjoining geometry shows an incident envelope above the grating and the calculated full transmitted intensity below it.

The incident extension is a geometric illustration; propagation is computed from the boundary onward.

**Notation:** use $U_{\mathrm{exit}}$ at the exit plane, $U_{\mathrm{ref}}$ at a general reference plane, and $C_m$ (or $C_{mp}$) for field coefficients. [Shared convention](0_0_Foundations_of_Scalar_Optics.ipynb#exit-field-notation).


## Numerical status — read before interpreting selected orders

**The full slider range has not been validated for quantitative work.** Tests found significant errors in some high-order and near-grazing cases. An example can look smooth, and show no warning, while still giving an inaccurate result. Check three separate questions:

1. **Full-field convergence:** resolution and padding convergence of the complete propagated boundary field in a declared observation region.
2. **Selected-band convergence:** convergence of the complex fields retained by the midpoint windows, including their near-grazing components.
3. **Order interpretation:** whether those filtered bands approximate isolated order contributions rather than overlapping coherent spectral replicas.

One can pass while another fails. The default was well behaved in the tested region, but an exposed high-order example ($\lambda=0.6$, $d=6$, $w_0=16$ µm, incidence $10^\circ$, pair $(-11,0)$) had **10.31% selected intensity bias at 64 samples/period with no warning**. Increasing to 256 samples/period reduced it to approximately 0.61%. Near grazing ($\lambda=1.1$, $d=4$, $w_0=16$ µm, incidence $10^\circ$, pair $(0,3)$), doubling the domain changed overlap by up to 0.08275. Strongly overlapping replicas can give a smooth overlap curve while the isolated-order interpretation remains unreliable.

For quantitative work vary sampling, domain width and filter boundaries independently; compare complex fields on matching physical coordinates. Check physically significant regions, not relative errors in nearly empty tails. A small edge fraction, Parseval identity or bounded overlap alone is insufficient. Spatial overlap is neither mutual coherence nor visibility. Keep the full-field, selected-band and order-interpretation checks separate. The numerical limitations described here remain; these examples do not validate the full slider range.

Numerical comparisons do not certify every setting. Check convergence for the chosen geometry and observable, and keep numerical error separate from the analytical model's assumptions.

:::{dropdown} Numerical audit record

The audit identifier is NH-7.3.1. It examines the FFT/ASM implementation and its runtime warnings. The tests above document the current numerical limitations.
:::


In [ ]:
from pathlib import Path
import sys
import numpy as np
from IPython.display import display

_support_dir = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / 'wave_optics' / '__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open this notebook within Grating_Physics.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))
from wave_optics.finite_grating_beams import FiniteGratingConfiguration, FiniteGratingExperiment, intensity_overlap
from wave_optics.finite_grating_beams_view import FiniteGratingBeamsExplorer


In [ ]:
if 'finite_grating' in globals():
    finite_grating.close()
finite_grating = FiniteGratingBeamsExplorer()
display(finite_grating.ui)


## 2. Generate finite diffraction-order fields

At the transparent openings, the boundary inherits the incident complex field; elsewhere it is zero:

$$U_{\mathrm{exit}}(x)=t(x)U_i(x).$$

Multiplication by the periodic mask replicates the incident spectrum around the familiar centers

$$k_{x,m}=k_{x,i}+mG,\qquad G=2\pi/d.$$

For infinite illumination these components are discrete harmonics. For finite Gaussian illumination, each replica has finite width. Its isolated Gaussian amplitude varies as $\exp[-w_0^2(k_x-k_{x,m})^2/4]$. Smaller $w_0$ therefore gives a broader spectrum. The total spectrum is a coherent sum of replicas; when they overlap, an observed maximum need not sit exactly on an ideal center.

**Try it:** reduce $w_0$ from 8 to 3 µm. Panel B keeps the ideal centers marked while the measured spectrum broadens. Read the warning if neighboring regions cease to be well isolated. Then change pitch: increasing $d$ brings the centers closer together, making isolation harder at the same radius. Reset before the propagation experiment.

This is the finite-aperture feature of classical diffraction: limiting the illuminated extent spreads angular content. It is experimentally realizable with a finite-width laser, whereas infinite illumination is a useful limiting model. The lesson uses a mask and scalar propagation, rather than reproducing a historical apparatus or assigning rigorous grating efficiencies. [1–3](#refs-appendix-2)

### Small calculation 1: envelope width and spectral width

Predict which radius gives the wider isolated Gaussian spectrum. These analytical widths describe one replica, not an exact fit to overlapping peaks of the full masked field.


In [ ]:
d_um = 4.0
G = 2*np.pi/d_um
for w0_um in (8.0, 4.0, 2.0):
    sigma_k_x = 1/w0_um  # Standard deviation of spectral intensity.
    midpoint_amplitude = np.exp(-(w0_um*G/4)**2)
    print(f'w0={w0_um:g} µm: sigma_kx={sigma_k_x:.3f} rad/µm; '
          f'isolated-replica amplitude at midpoint={midpoint_amplitude:.3e}')


:::{dropdown} Answer 1

The spectral-intensity standard deviations are 0.125, 0.250 and 0.500 rad/µm. Narrower illumination broadens the spectrum without moving the ideal centers. The midpoint number estimates one Gaussian replica’s amplitude relative to its peak; it is a warning diagnostic, not a measured cross-talk probability.

:::

## 3. Propagate the finite transmitted field

The angular-spectrum method Fourier-transforms the boundary, advances each component, then reconstructs the complex field. For transmission into $z<0$, use

$$k_z=-\sqrt{k^2-k_x^2},\qquad k=2\pi/\lambda.$$

The square-root branch gives negative real $k_z$ for propagation and negative imaginary $k_z$ for evanescence. With $z=-s$, an evanescent component has $e^{ik_zz}=e^{-\kappa s}$, not exponential growth. The existing propagation utility supplies exactly this multiplier when its positive distance argument is interpreted as depth $s$.

Panel A’s lower map is calculated from the complete sampled transmitted field. Different angular portions move outward with depth, while each finite contribution also diffracts and spreads. A beam’s central direction is informative, but its arrow alone cannot describe that field. The other maps show the selected components introduced next; they do not include every part of the full output.

**Predict, then move the detector:** will increasing depth remove either field everywhere, or change where their intensities are appreciable? The green lines mark one detector plane across the maps. The displayed spatial window is only a crop of a much wider FFT computation. A field leaving the picture has not necessarily left the numerical domain.

Numerical padding separates this physical spreading from FFT periodic wraparound. The readout reports grid spacing, spectral resolution and intensity near the outer numerical edges. A large edge fraction is a reason to enlarge the domain and compare results, not evidence of a physical returning beam. Optional background gives the checks used here.

## 4. Isolate two approximate order contributions

Choose two nonzero propagating order centers in the dropdown. Each numerical window extends halfway toward the neighboring ideal centers: $k_{x,m}-G/2\leq k_x<k_{x,m}+G/2$. It is also clipped to $|k_x|<k$, so the retained contributions propagate. Blue and orange identify the same selections in every panel.

These disjoint windows are an explicit, reproducible definition of approximate order beams, not exact harmonics of an infinite grating. They retain the complex spectrum and introduce no random phases. Hard window edges can create weak spatial tails. Near grazing, clipping changes the beam further; heed the displayed warning. Strongly overlapping replicas cannot be assigned uniquely to orders by this procedure.

The individual fields and their coherent sum are

$$U_m(x,z),\quad U_p(x,z),\qquad U_{\mathrm{total}}=U_m+U_p,$$

with interference contribution

$$I_{\mathrm{cross}}=2\operatorname{Re}[U_m^*U_p],\qquad
|U_m+U_p|^2=|U_m|^2+|U_p|^2+I_{\mathrm{cross}}.$$

Compare the three selected-field maps, then inspect the signed cross-term map. Positive regions reinforce; negative regions reduce the summed individual intensities. The total intensity remains nonnegative. The plotted pair is only the retained spectrum, so its sum need not reproduce the complete transmitted field.

### Small calculation 2: reconstruct the retained field

Predict whether filtering the union of two disjoint windows agrees with summing their propagated fields. This fixed calculation is independent of the sliders.


In [ ]:
experiment = FiniteGratingExperiment()
sample = experiment.selected_fields((0, 1), depths=np.array([0., 40., 80.]))
retained_spectrum = np.sum(sample['spectra'], axis=0)
retained_field = experiment.propagate_spectrum(retained_spectrum, sample['depth'])
print('Retained-field reconstruction error:', np.max(abs(retained_field-sample['total'])))
print('Intensity identity error:', np.max(abs(abs(sample['total'])**2
    -abs(sample['fields'][0])**2-abs(sample['fields'][1])**2-sample['cross'])))


:::{dropdown} Answer 2

Both errors should be at floating-point rounding level. Propagation is linear, and the windows are disjoint. This checks the retained two-window field, not its equality to the full unfiltered output.

:::

## 5. Spatial overlap and observable fringes

At each detector depth, compare the blue and orange intensity profiles with their coherent sum. Near the grating they occupy much of the same region. Farther away their different central directions can separate their envelopes. Spreading also changes their widths, so separation and diffraction act together; overlap need not decrease monotonically for every parameter choice.

Panel D measures a normalized spatial intensity overlap:

$$\mathcal O(z)=\frac{\int |U_m(x,z)||U_p(x,z)|\,dx}
{\sqrt{\int |U_m|^2dx\int |U_p|^2dx}}.$$

It lies between zero and one for nonzero fields. It uses the whole padded detector coordinate, not just the visible crop. These integrals are transverse intensity per unit length along invariant $y$, not total optical power. Zero norm makes the diagnostic undefined.

**This overlap is not mutual-coherence magnitude.** The two deterministically filtered fields retain their phase relationship. They can remain fully mutually coherent wherever both are nonzero while having very little integrated spatial overlap. Where either field is negligible, the local product $U_m^*U_p$ and observable interference contribution are small.

For the starting pair, the zeroth-order center stays near $x=0$, while the first-order center moves approximately $s\tan\theta_1$, with $\sin\theta_1=\lambda_0/(nd)=\lambda/d=0.25$. At depth 60 µm this predicts about 15.5 µm of separation. Compare that distance with the envelope radius before reading the overlap curve. The calculated profiles include their finite angular width and spreading, so this central-direction estimate is a guide rather than a replacement for propagation.

A useful distinction is that the signed cross term can cancel when integrated across many bright and dark fringes, even while interference is locally strong. The overlap diagnostic instead integrates the nonnegative product of field magnitudes. It measures where modulation could occur; it does not measure its signed sum, resolve its fringes or determine coherence from intensity alone.

**Try it:** start with $(0,+1)$ and move from depth zero to 60 µm. Watch profile separation and the overlap marker together. Increase $w_0$, reset, then reduce it. A wider incident envelope begins with narrower angular peaks, but its spatial extent also delays separation. Explain the changes rather than assuming one universal overlap curve. Do not impose the infinite-grating period $d/|p-m|$ on every part of these filtered, diffracting fields.

### Small calculation 3: overlap is not correlation

Predict the overlap of identical fields and of separated sampled fields. Multiplying either field by a constant phase cannot change this intensity-overlap diagnostic.


In [ ]:
x_um = np.linspace(-40., 40., 4001)
dx_um = x_um[1]-x_um[0]
U_m = np.exp(-x_um**2/8.**2)
U_p = np.exp(-(x_um-20.)**2/8.**2)
print('Identical envelopes:', intensity_overlap(U_m, U_m, dx_um))
print('Separated envelopes:', intensity_overlap(U_m, U_p, dx_um))
print('Same envelopes, extra phase:', intensity_overlap(U_m, U_p*np.exp(1j*np.pi/2), dx_um))


:::{dropdown} Answer 3

The values are 1 and approximately $e^{-20^2/(2\cdot8^2)}=0.04394$. The phase-shifted case has the same overlap. This says how much the intensity envelopes coincide, not how stable their relative phase is under an averaging procedure.

:::

## Final checkpoint

1. Why do finite illumination and a finite plotting window mean different things?
2. What determines the ideal centers and the widths of the spectral replicas?
3. Why must the evanescent branch decay toward negative $z$?
4. What exactly does the numerical order filter retain?
5. Can low spatial overlap establish low mutual coherence?
6. Why might a later experiment use mirrors?

:::{dropdown} Checkpoint answers

1. Illumination changes the boundary field; cropping only changes what is displayed.
2. Grating pitch and incidence set the centers; the incident envelope sets replica width.
3. It is the bounded outgoing solution; the opposite sign grows away from the boundary.
4. Complex spectral samples in explicit midpoint cells, clipped to propagating frequencies; it does not uniquely identify overlapping replicas.
5. No. The diagnostic discards phase, while coherence describes field correlation under a specified average.
6. Mirrors can redirect separated finite fields into a common region. They are unnecessary for the spatial overlap of ideal extended plane waves.

:::

**Return to the main sequence:** [7.3](7_3_Finite_Diffraction_Beams_and_Experimental_Recombination.ipynb) treats prescribed finite-beam overlap, [7.4](7_4_Diffraction_Kinematics_and_Complex_Order_Coefficients.ipynb) distinguishes directions from complex strengths, and the [closing synthesis](Closing_0_From_Diffraction_to_Imaging.ipynb) completes Series 1. Mirror-assisted recombination is an optional extension, not a remaining requirement. Compare [7.2](7_2_Interference_of_Diffraction_Orders_from_a_Grating.ipynb) for ideal extended orders.


:::{dropdown} Optional: numerical method, limitations and handoff

The periodic binary transmission comes from `binary_transmission_field`; propagation reuses `scalar_propagation.transfer_function`. The grid has 64 samples per period, with cell centers offset by half a step to avoid sampling mask discontinuities. The domain is a power-of-two FFT grid at least 512 µm wide and also at least $12w_0+5s_{\max}$. No physical aperture is imposed at its edges; the Gaussian is already negligible there. Fourier spectra are raw NumPy FFT weights. For a continuous-transform density include `dx` and the sampling-origin phase; propagation itself uses the consistent FFT pair.

Output maps use 81 depth samples from 0 to 80 µm. The detector control samples those same planes. Changing detector depth reuses the complex-field maps; changing geometry or the selected pair recomputes the required fields. Figure artists and widget observers are retained. Reset restores all starting parameters; rerunning the display cell closes its previous owned explorer. Run `finite_grating.close()` when finished.

Order windows are half-open midpoint cells, clipped at the propagating cutoff. The midpoint amplitude diagnostic is $\exp[-(w_0G/4)^2]$ for one Gaussian replica. Values above 0.01 trigger an overlap warning. This is conservative guidance, not an error bound on spectral attribution. Even nonzero integer orders vanish in the ideal 50%-open mask and are not offered as strong isolated beams.

The finite FFT domain remains periodic. Edge intensity is a useful warning, not a proof against every wraparound artifact. Quantitative work must compare a wider padded grid, finer spatial sampling and the region of interest. Dedicated tests compare padding and spatial resolution for representative configurations. Sharp-mask sampling converges with resolution and can leave weak high-frequency leakage.

For retained propagating spectra, the full-domain transverse intensity integrals remain constant under ASM. The full masked field also includes evanescent content, whose squared magnitude decays; its total transverse intensity integral need not stay constant near the boundary. Normal scalar flux instead weights spectral intensity by $\operatorname{Re}(-k_z)/k$ and is conserved in the homogeneous output medium. Neither quantity alone supplies a rigorous electromagnetic grating efficiency.

`finite_grating.model.selected_fields(pair, depths=...)` returns individual complex arrays `[order, depth, x]`, their sum, full unfiltered field, signed cross term, windows, spectra, signed wavevectors, coordinates, overlap and diagnostics. The `metadata` dictionary records the time convention, output half-space, units and configuration. Arrays preserve phase for Stage 7.3B; they have not been reflected or physically recombined. The model is scalar and invariant along $y$: it does not include finite extent in both transverse directions, polarization-dependent scattering, partial-coherence integration or optical imaging.

:::



(refs-appendix-2)=

## Further Reading

(ref-appendix-2-bandlimited)=

- K. Matsushima and T. Shimobaba, “Band-Limited Angular Spectrum Method for Numerical Simulation of Free-Space Propagation in Far and Near Fields,” *Optics Express*, vol. 17, pp. 19662–19673, 2009. [Online]. Available: [PDF](https://kansai-u.repo.nii.ac.jp/record/11127/files/KU-1100-20091015-50.pdf). doi: 10.1364/OE.17.019662.

  **Reading notes:** **Access:** Free institutional-repository full text; copyright retained. **Read for:** Read the sampling discussion; this notebook does not implement that algorithm.

(ref-appendix-2-mansuripur)=

- M. Mansuripur, “A Tutorial on the Classical Theories of Electromagnetic Scattering and Diffraction,” *Nanophotonics*, vol. 10, pp. 315–342, 2021; arXiv version submitted 2020. [Online]. Available: [preprint](https://arxiv.org/abs/2009.06706).

  **Reading notes:** **Access:** Free arXiv manuscript; not public domain. **Read for:** Optional advanced reading: scalar diffraction and its electromagnetic assumptions.

## Reference notes

Freely accessible works are not necessarily in the public domain.



## Course connections

(course-appendix-2-1)=

[0.4](0_4_Propagation_of_Scalar_Optical_Fields.ipynb): Fourier decomposition, ASM branch choices, finite FFT domains and propagation limits.

(course-appendix-2-2)=

[0.7](0_7_Ideal_Binary_Masks_vs_Physical_Gratings.ipynb): prescribed thin masks versus physical grating scattering; reference-plane complex coefficients.

(course-appendix-2-3)=

[1.2](1_2_Ideal_Plane_Waves_versus_Finite_Optical_Beams.ipynb) and [7.1](7_1_Interference_and_Mutual_Coherence_of_Two_Plane_Waves.ipynb): finite-beam envelopes, interference and measurement-dependent visibility. [2.3](2_3_Allowed_Diffraction_Orders.ipynb) gives the order-center geometry.
